# Lab 03: Prompt Caching

## Overview

In this notebook, we enable **prompt caching** to cut the cost of the *input* side. Caching stores the processed system prompt and tool definitions so repeated requests reuse them at a steep discount (cached reads are billed at ~10% of the normal input rate — a 90% discount).

**What you'll learn:**
- How to configure system prompt caching
- How to enable tool definition caching
- How to read cache hits/writes in Langfuse
- How caching trades a small first-request premium for large savings on reuse

**Optimizations in this notebook:**
- `SystemContentBlock` with `cachePoint` (system prompt caching — provider-agnostic)
- `cache_tools="default"` on BedrockModel (tool definition caching)

## Prerequisites

- Completed Labs 01-02

## Workshop Journey

```
01 Baseline → 02 Quick Wins → [03 Caching] → 04 Routing → 05 Guardrails → 06 Skills + Gateway → 07 Evaluations
                                   ↑
                              You are here
```

## Step 1: Setup

In [1]:
from __future__ import annotations

import json
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(override=True)

import boto3
from bedrock_agentcore_starter_toolkit import Runtime

region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
control_client = boto3.client("bedrock-agentcore-control", region_name=region)
data_client = boto3.client("bedrock-agentcore", region_name=region)
agentcore_runtime = Runtime()

print(f"Region: {region}")
print(f"Langfuse Host: {os.environ.get('LANGFUSE_BASE_URL', 'Not set')}")

/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


Region: us-east-1
Langfuse Host: https://d1fnzg75c19u2d.cloudfront.net


## Step 2: Understanding Prompt Caching

### What is Prompt Caching?

Prompt caching stores the processed *prefix* of a request (the parts that don't change between calls) so subsequent requests skip recomputing it. This notebook enables **two cache points**:

| Cache Type | What's Cached | Configuration |
|------------|---------------|---------------|
| **System Prompt** | The full system prompt | `SystemContentBlock` with `cachePoint` |
| **Tool Definitions** | All tool schemas and descriptions | `cache_tools="default"` on BedrockModel |

Together these make up the cached prefix. In this lab's run that prefix is **~2,839 tokens** — you'll see exactly that number show up as `cacheWriteInputTokens` on the first call and `cacheReadInputTokens` on the warm ones.

### Minimum Token Requirement

Caching only activates once the prefix before a cache checkpoint meets the model's minimum. Below it, inference still succeeds but the prefix is silently **not** cached (`cacheWriteInputTokens`/`cacheReadInputTokens` stay 0, no error). The minimum is **per model** — current Bedrock Claude values ([AWS docs](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html)):

| Model | Min tokens / checkpoint | Supported TTL |
|---|---|---|
| **Claude Sonnet 4.6** *(this lab)* | **1,024** | 5 min |
| Claude Sonnet 4.5 | 4,096 | 5 min, 1 hour |
| Claude Haiku 4.5 | 4,096 | 5 min, 1 hour |
| Claude Opus 4.5 | 4,096 | 5 min, 1 hour |
| Claude Opus 4.6 | 4,096 | 5 min |
| Claude Opus 4.7 | 4,096 | 5 min, 1 hour |
| Claude Opus 4.8 | 4,096 | 5 min, 1 hour |
| Claude 3.7 Sonnet | 1,024 | 5 min |

> **Watch the minimum when you swap models.** Sonnet 4.6 (this lab) caches from 1,024 tokens, but every current **Opus 4.x** — 4.5, 4.6, 4.7, 4.8 — needs **4,096**, as do Sonnet 4.5 and Haiku 4.5. A prompt that caches fine on Sonnet 4.6 could silently stop caching after a move to Opus. That's why Lab 02 kept the prompt large rather than trimming it.
>
> *(Platform caveat: these are the **Bedrock** minimums from the Bedrock model cards. Anthropic's first-party API publishes different numbers for some models — e.g. it lists Opus 4.8 at 1,024 — so always check the docs for the platform you're actually running on. On Bedrock, Opus 4.7 and 4.8 are 4,096.)*

### The write premium (why the first call costs more)

Caching isn't free on the *first* request. Writing to the cache is billed at a **premium** over normal input tokens (~1.25× for the standard 5-minute TTL); cached **reads** are then ~10% of the normal rate. So caching only pays off when the prefix is reused — it breaks even at roughly **two requests** within the TTL window, and saves more from there.

### System Prompt Caching

Use `SystemContentBlock` with a cache point at the end:

```python
from strands.types.content import SystemContentBlock

system_prompt = [
    SystemContentBlock(text=SYSTEM_PROMPT_TEXT),        # must clear the model's minimum
    SystemContentBlock(cachePoint={"type": "default"})  # cache checkpoint
]

agent = Agent(system_prompt=system_prompt)
```

### Tool Definition Caching

Enable tool caching via `BedrockModel`:

```python
model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-6",
    cache_tools="default"  # cache all tool definitions
)
```

> **Ordering note:** Bedrock chains the cache sections `tools → system → messages`, and the token minimum is checked against their **combined** size — so caching `tools` + `system` together (as we do) is what clears the 1,024 floor and keeps the stable content ahead of the variable user message.

### Cache Metrics in Langfuse

- `cacheWriteInputTokens` — tokens written to cache (the cold first request, billed at the write premium)
- `cacheReadInputTokens` — tokens read from cache (subsequent requests within the TTL; the default TTL is **5 minutes** and resets on each hit)

Pricing varies by model — see [Amazon Bedrock Pricing](https://aws.amazon.com/bedrock/pricing/) for current cache read/write rates.

### How it works, end to end

<img src="images/how%20prompt%20caching%20works.png" alt="Prompt caching flow: check for cached prefix, write on miss or read on hit, with a 5-minute TTL" width="620">

Reading the flow: every request's marked prefix is checked against the cache. On a **miss**, the model processes the prefix and — *if it clears the token minimum* — writes a cache checkpoint (otherwise it just continues, uncached). On a **hit**, the model skips straight to the last cached checkpoint and resumes, which is where the latency and cost savings come from. Each hit resets the 5-minute TTL; if nothing hits within that window, the entry is evicted and the next request pays the write again.

For a deeper walkthrough with worked pricing examples, see AWS's [Effectively use prompt caching on Amazon Bedrock](https://aws.amazon.com/blogs/machine-learning/effectively-use-prompt-caching-on-amazon-bedrock/).

In [2]:
# Review the caching configuration in v3 agent
agent_file = Path("agents/v3_caching.py")
print(agent_file.read_text())

"""
V3 Caching Agent - Same as v2 + prompt caching.
- All v2 optimizations (structured prompt, max_tokens, stop_sequences, low temperature)
- System prompt caching with SystemContentBlock + cachePoint (provider-agnostic)
- Tool definition caching with cache_tools="default" on BedrockModel
- Note: System prompt must be 1,024+ tokens for caching to activate
"""

from __future__ import annotations

import os

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from strands.types.content import SystemContentBlock

from utils.agent_config import MODEL_SONNET, SYSTEM_PROMPT_TEXT, setup_langfuse_telemetry
from utils.tools import get_product_info, get_return_policy, get_technical_support, web_search

setup_langfuse_telemetry()

app = BedrockAgentCoreApp()

# Provider-agnostic caching using SystemContentBlock
SYSTEM_PROMPT = [
    SystemContentBlock(text=SYSTEM_PROMPT_TEXT),
  

## Step 3: Deploy the Agent with Prompt Caching

In [3]:
agent_name = "customer_support_v3_caching"
agent_file = str(Path("agents/v3_caching.py").absolute())
requirements_file = str(Path("requirements-for-agentcore.txt").absolute())

# Clean up any existing build files from previous labs
for f in ["Dockerfile", ".dockerignore", ".bedrock_agentcore.yaml"]:
    p = Path(f)
    if p.exists():
        p.unlink()
        print(f"Removed existing: {f}")

print(f"Configuring agent: {agent_name}")
agentcore_runtime.configure(
    entrypoint=agent_file,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file=requirements_file,
    region=region,
    agent_name=agent_name,
)

Entrypoint parsed: file=/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/agents/v3_caching.py, bedrock_agentcore_name=v3_caching
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: customer_support_v3_caching


Removed existing: Dockerfile
Removed existing: .dockerignore
Removed existing: .bedrock_agentcore.yaml
Configuring agent: customer_support_v3_caching


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: 
/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/Dockerfile

Generated .dockerignore: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.dockerignore
Setting 'customer_support_v3_caching' as default agent
Bedrock AgentCore configured: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/Dockerfile'), dockerignore_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.dockerignore'), runtime='None', runtime_type=None, region='us-east-1', account_id='739907928487', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

In [4]:
# Modify Dockerfile for Langfuse
dockerfile_path = Path("Dockerfile")
if dockerfile_path.exists():
    content = dockerfile_path.read_text()
    # Replace opentelemetry-instrument wrapper with direct python call
    # Keep the correct module path using regex
    if "opentelemetry-instrument" in content:
        import re

        content = re.sub(
            r'CMD \["opentelemetry-instrument", "python", "-m", "([^"]+)"\]', r'CMD ["python", "-m", "\1"]', content
        )
        dockerfile_path.write_text(content)
        print("Dockerfile modified for Langfuse")
    else:
        print("Dockerfile already configured or using different format")
else:
    print("Dockerfile not found - will be created during deployment")

Dockerfile modified for Langfuse


In [5]:
env_vars = {
    "LANGFUSE_BASE_URL": os.environ.get("LANGFUSE_BASE_URL"),
    "LANGFUSE_PUBLIC_KEY": os.environ.get("LANGFUSE_PUBLIC_KEY"),
    "LANGFUSE_SECRET_KEY": os.environ.get("LANGFUSE_SECRET_KEY"),
    "PYTHONUNBUFFERED": "1",
}

print("Deploying to AgentCore Runtime...")
launch_result = agentcore_runtime.launch(env_vars=env_vars, auto_update_on_conflict=True)
agent_arn = launch_result.agent_arn
print(f"Agent deployed: {agent_arn}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'customer_support_v3_caching' to account 739907928487 (us-east-1)
Generated image tag: 20260602-042815-407
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: customer_support_v3_caching


Deploying to AgentCore Runtime...


ECR repository available: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v3_caching
Getting or creating execution role for agent: customer_support_v3_caching
Using AWS region: us-east-1, account ID: 739907928487
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-9236411bd3


✅ Reusing existing ECR repository: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v3_caching


✅ Reusing existing execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-9236411bd3
Execution role available: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-9236411bd3
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: customer_support_v3_caching
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-9236411bd3
Reusing existing CodeBuild execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-9236411bd3
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: customer_support_v3_caching/source.zip
Updated CodeBuild project: bedrock-agentcore-customer_support_v3_caching-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.3s
🔄 PROVISIONING started (total: 2s)
✅ PROVISIONING completed in 6.3s
🔄 DOWNLOAD_SO

Agent deployed: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/customer_support_v3_caching-spXyyp5y1x


In [6]:
# Save the agent ARN for later use
agent_arn = launch_result.agent_arn
print(f"Agent ARN: {agent_arn}")

# Grant the auto-created execution role the shared runtime permissions
# (SSM KB lookup + Bedrock Retrieve + ApplyGuardrail) used across all labs.
from utils.runtime_helpers import ensure_runtime_permissions

ensure_runtime_permissions(region)

Agent ARN: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/customer_support_v3_caching-spXyyp5y1x
Granted customer-support runtime permissions to AmazonBedrockAgentCoreSDKRuntime-us-east-1-9236411bd3


## Step 4: Test Caching Behavior

Run the standard test prompts and watch the cache metrics in Langfuse.

**What to look for:**
- `Cache Write Tokens` > 0 on the **first** request — the prefix being written to cache (billed at the write premium).
- `Cache Read Tokens` > 0 on **later** requests — the prefix served from cache at ~10% of the input rate.
- The cache is warm if requests land within the 5-minute TTL, so depending on timing you may see reads on the very first row too.

One quirk you'll notice below: tool-calling turns show **roughly double** the cache reads of the no-tool turn. That's because a tool call makes the agent loop hit the model twice (once to decide the tool, once to answer with the result), and the cached prefix is read on **each** model call within the trace.

In [7]:
def invoke_agent(prompt):
    """Invoke the agent via AgentCore API."""
    response = data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return json.loads(response["response"].read().decode("utf-8"))

In [8]:
# Import Langfuse metrics helper
from utils.langfuse_metrics import (
    clear_metrics,
    collect_metric,
    get_latest_trace_metrics,
    print_metrics,
    print_metrics_table,
)

# Clear any previously collected metrics
clear_metrics()

# Standard test prompts - same across all notebooks for consistent comparison
TEST_PROMPTS = [
    # Single tool: get_return_policy
    ("Return Policy", "What is your return policy for laptops?"),
    # Single tool: get_product_info
    ("Product Info", "Tell me about your smartphone options"),
    # Single tool: get_technical_support (Bedrock KB)
    ("Technical Support", "My laptop won't turn on, can you help me troubleshoot?"),
    # Multi-tool: get_product_info + get_return_policy
    ("Multi-part Question", "I want to buy a laptop. What are the specs and what's the return policy?"),
    # No tool: General greeting
    ("General Question", "Hello! What can you help me with today?"),
]

# Run all tests and collect metrics
# Cache behavior: First request after deployment writes to cache, subsequent requests read from cache
# If cache is already warm (within 5-min TTL), all requests will show cache reads
for i, (test_name, prompt) in enumerate(TEST_PROMPTS):
    print("=" * 60)
    print(f"Test {i + 1}: {test_name}")
    print("=" * 60)

    response = invoke_agent(prompt)
    print(response)

    # Fetch and collect metrics
    metrics = get_latest_trace_metrics(
        agent_name="customer-support-v3-caching",
        wait_seconds=5,
        max_retries=5,
        timeout_seconds=120,
    )
    print_metrics(metrics, test_name)
    collect_metric(metrics, test_name)

Test 1: Return Policy
Here's a summary of our laptop return policy:

- **Return Window:** 30 days from the date of purchase
- **Condition Requirements:** Must be in original packaging with all accessories included and no physical damage
- **Restocking Fee:** None for defective items; 15% fee applies for change-of-mind returns
- **Return Shipping:** Free if the laptop is defective; customer covers shipping for change-of-mind returns
- **Refund Timeline:** 7–10 business days after the returned item is inspected
- **How to Return:** You can initiate a return through our **online RMA portal** or bring the laptop to any **TechMart store** in person
- **Warranty:** All laptops come with a **1-year manufacturer warranty**, and extended warranty options are also available

Would you like help starting a return, or do you have any other questions about our laptop policies or products?

- **category:** policy
- **confidence:** high

                    LANGFUSE METRICS
  Test:          Return Po

In [9]:
# Print summary table
print_metrics_table()

# Save metrics for comparison in later notebooks
from utils.langfuse_metrics import save_metrics

save_metrics("v3")


                                  METRICS SUMMARY
               Test Latency    Cost Input Output Cache Read Tokens Cache Write Tokens
      Return Policy   5.86s $0.0076   863    277             2,839              2,839
       Product Info   3.01s $0.0056   910     77             5,678                  0
  Technical Support   8.40s $0.0097 1,302    273             5,678                  0
Multi-part Question   9.70s $0.0139 1,225    568             5,678                  0
   General Question   3.97s $0.0047   332    191             2,839                  0
---------------------------------------------------------------------------------------------------------
  TOTALS: Latency(avg): 6.18s | Cost: $0.0415 | Input: 4,632 | Output: 1,386
          Cache Read Tokens: 22,712 | Cache Write Tokens: 2,839

Metrics saved as 'v3' → .lab_metrics.json


## Step 5: Compare with v2 (Quick Wins)

Enter your metrics from Lab 02 (v2 quick wins) to compare cost, latency, and token usage.

In [10]:
from utils.langfuse_metrics import calculate_totals_from_collected, load_metrics, print_comparison

# Load metrics from Lab 02 (saved automatically when you ran print_metrics_table())
v2 = load_metrics("v2")

# Or enter manually if Lab 02 metrics weren't saved:
# v2 = {"total_cost": 0.0858, "avg_latency": 7.30, "total_input_tokens": 19920, "total_output_tokens": 1737}

# Print comparison (current metrics auto-calculated from collected)
print_comparison(
    prev_name="v2 (Quick Wins)",
    curr_name="v3 (Caching)",
    prev_cost=v2["total_cost"],
    prev_latency=v2["avg_latency"],
    prev_input_tokens=v2["total_input_tokens"],
    prev_output_tokens=v2["total_output_tokens"],
)

# Show cache-specific metrics (unique to v3)
totals = calculate_totals_from_collected()
print("\nCache Metrics (v3 only):")
print(f"  Cache Read Tokens:  {totals['total_cache_read_tokens']:,}")
print(f"  Cache Write Tokens: {totals['total_cache_write_tokens']:,}")

Loaded 'v2' metrics: cost=$0.1047, latency=5.08s, input=30,183, output=946
  V2 (QUICK WINS) vs V3 (CACHING) COMPARISON
Metric                  v2 (Quick Wins)       v3 (Caching)       Change
----------------------------------------------------------------------
Total Cost           $           0.1047 $           0.0415       -60.4%
Avg Latency (s)                    5.08               6.18       +21.8%
Input Tokens                     30,183              4,632       -84.7%
Output Tokens                       946              1,386       +46.5%

Result: 60.4% cost reduction, -21.8% latency increase

Cache Metrics (v3 only):
  Cache Read Tokens:  22,712
  Cache Write Tokens: 2,839


### Reading the comparison

The headline: **cost drops sharply** and **input tokens fall by most of the cached prefix** — exactly what caching is for. Almost the entire system-prompt + tools prefix now reads from cache at a fraction of the price instead of being reprocessed every call. (Your exact percentages will vary run to run — watch the direction, not the decimals.)

Two things that can look "wrong" but aren't:

- **Latency might tick *up* in a small run.** It's only 5 prompts, and a cold cache write, a model cold start, or one slow tool call easily swings a tiny sample. Caching *reduces* time-to-first-token at scale (AWS quotes up to ~85% latency reduction on cache-heavy workloads), but on five mixed requests the noise dominates. Watch cost here, not latency.
- **Output tokens may move in either direction.** Caching only touches the input prefix — it has nothing to do with how much the model generates. Any change is just normal run-to-run variation in answer length, not a caching effect.

The real lesson: caching is an **input-side** lever. It makes each repeated request cheaper without changing the agent's behavior at all.

## Summary

In this notebook, we enabled **two cache points** on top of the v2 optimizations:

| Cache Type | Configuration |
|------------|---------------|
| **System Prompt** | `SystemContentBlock` + `cachePoint` |
| **Tool Definitions** | `cache_tools="default"` |

Together they cached a **~2,839-token prefix** — written once (at the write premium), then read at ~10% of the input rate on every reuse, which is where the ~60% cost drop comes from.

```python
from strands.types.content import SystemContentBlock

# 1. System prompt caching
system_prompt = [
    SystemContentBlock(text=SYSTEM_PROMPT_TEXT),  # must clear the model's token minimum
    SystemContentBlock(cachePoint={"type": "default"})
]

# 2. Tool definition caching
model = BedrockModel(
    cache_tools="default",
    ...
)

agent = Agent(model=model, system_prompt=system_prompt)
```

**Key observations:**
- First request after the cache expires *writes* to cache (premium); later requests *read* (90% discount).
- The 5-minute TTL refreshes on each cache hit, so steady traffic keeps the cache warm.
- The prefix must clear the model's token minimum (1,024 for Sonnet 4.6) or caching silently no-ops.
- Caching is an **input-side** win — it doesn't change output length, latency-per-token, or answer quality.

**When caching pays off:**
- Consistent traffic (repeated requests inside the TTL window).
- Static, sizeable system prompts and tool definitions.
- High-volume production workloads — the more reuse, the bigger the saving.
- *Not* worth it for one-off calls: the write premium with no reuse is a net loss.

**Pricing:** see [Amazon Bedrock Pricing](https://aws.amazon.com/bedrock/pricing/) for current cache read/write rates.

---

**Next:** In Lab 04, we'll explore **model routing** to send simple queries to a cheaper model.

**Next notebook:** [04-llm-routing.ipynb](./04-llm-routing.ipynb)

---

## Cleanup

To delete the agent deployed in this notebook, uncomment and run the following code.

In [11]:
# # Uncomment to delete resources created in this lab
# agentcore_runtime.destroy(delete_ecr_repo=True)
# print(f"Deleted agent and ECR repository: {agent_name}")